# CS2 职业比赛胜负预测系统

本项目使用机器学习方法预测 CS2 职业比赛结果。

## 项目思路
1. **数据准备** – 模拟真实 CS2 职业比赛统计数据
2. **探索性数据分析 (EDA)** – 可视化关键特征
3. **特征工程** – 提取队伍近期表现、排名差异、头对头战绩等特征
4. **多模型训练** – 逻辑回归、随机森林、XGBoost、LightGBM
5. **超参数调优** – 使用随机搜索与交叉验证
6. **集成模型** – Stacking 集成提升准确率
7. **评估与可视化** – Accuracy、AUC-ROC、混淆矩阵
8. **预测接口** – 输入两支队伍即可预测比赛结果

In [ ]:
# ── 依赖库导入 ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               StackingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve,
                              ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 中文字体支持
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('所有依赖库加载完毕 ✅')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

## 1. 数据生成
以下函数模拟 CS2 职业联赛中的 **队伍统计数据** 和 **历史比赛记录**，
涵盖了 HLTV、ESL 等平台常见的关键指标。

In [ ]:
# ── 1.1 职业队伍基础信息 ────────────────────────────────────────────────────
TEAMS = [
    ('Natus Vincere', 1),  ("G2 Esports", 2),     ('FaZe Clan', 3),
    ('Heroic', 4),          ('ENCE', 5),             ('Liquid', 6),
    ('Cloud9', 7),          ('MOUZ', 8),             ('BIG', 9),
    ('Astralis', 10),       ('Vitality', 11),        ('NIP', 12),
    ('Complexity', 13),     ('OG', 14),              ('fnatic', 15),
    ('FURIA', 16),          ('paiN', 17),            ('Imperial', 18),
    ('9z', 19),             ('Apeks', 20),
]

def generate_team_stats(teams, seed=42):
    """为每支队伍生成稳定的基础统计特征，排名越靠前实力越强。"""
    rng = np.random.default_rng(seed)
    records = []
    n = len(teams)
    for name, rank in teams:
        # 将排名映射到 [0,1] 的实力系数 (1 = 最强)
        strength = 1.0 - (rank - 1) / n
        noise = rng.normal(0, 0.04)
        records.append({
            'team':              name,
            'rank':              rank,
            'rating':            round(1.05 + strength * 0.35 + noise, 3),      # HLTV Rating 2.0
            'kd_ratio':          round(0.90 + strength * 0.30 + rng.normal(0, 0.03), 3),
            'kast':              round(0.68 + strength * 0.10 + rng.normal(0, 0.02), 3),  # Kill/Assist/Survive/Trade
            'adr':               round(68 + strength * 18 + rng.normal(0, 2), 1),         # Average Damage per Round
            'opening_kill_rate': round(0.47 + strength * 0.08 + rng.normal(0, 0.02), 3),  # 首个击杀率
            'clutch_rate':       round(0.35 + strength * 0.15 + rng.normal(0, 0.02), 3),  # 局部残局胜率
            'map_pool_depth':    int(round(4 + strength * 4 + rng.normal(0, 0.5))),       # 地图池深度 (1-7)
            'flash_success':     round(0.30 + strength * 0.12 + rng.normal(0, 0.02), 3),  # 闪光弹成功率
        })
    df = pd.DataFrame(records)
    # 保证 map_pool_depth 在合理范围
    df['map_pool_depth'] = df['map_pool_depth'].clip(1, 7)
    return df

team_stats = generate_team_stats(TEAMS)
print('队伍统计数据：')
team_stats.head()

In [ ]:
# ── 1.2 模拟历史比赛记录 ────────────────────────────────────────────────────
def simulate_match(t1, t2, team_df, rng):
    """
    模拟单场比赛，根据两队实力差异确定胜负概率。
    返回 1 = team1 胜，0 = team2 胜。
    """
    s1 = team_df.loc[team_df['team'] == t1, 'rating'].values[0]
    s2 = team_df.loc[team_df['team'] == t2, 'rating'].values[0]
    # Elo 风格概率计算
    prob_t1 = 1 / (1 + 10 ** ((s2 - s1) / 0.15))
    return int(rng.random() < prob_t1)


def generate_match_history(team_df, n_matches=3000, seed=42):
    """生成完整的历史比赛数据集，包含丰富的特征列。"""
    rng = np.random.default_rng(seed)
    teams = team_df['team'].tolist()
    stats_map = team_df.set_index('team').to_dict('index')  # 快速查询

    # 先模拟所有比赛结果，用于计算近期胜率和头对头
    raw_matches = []
    for _ in range(n_matches):
        idx = rng.choice(len(teams), size=2, replace=False)
        t1, t2 = teams[idx[0]], teams[idx[1]]
        outcome = simulate_match(t1, t2, team_df, rng)
        raw_matches.append((t1, t2, outcome))

    # ── 构造特征 ──
    WINDOW = 15  # 近期胜率窗口大小
    h2h_wins = {}    # (t1, t2) -> t1 wins count
    h2h_total = {}   # (t1, t2) -> total matches
    recent_results = {t: [] for t in teams}   # 每队最近 WINDOW 场结果

    rows = []
    for t1, t2, outcome in raw_matches:
        key = tuple(sorted([t1, t2]))
        h2h_wins.setdefault(key, {t1: 0, t2: 0})
        h2h_total[key] = h2h_total.get(key, 0)

        # 计算近期胜率（使用当前窗口内数据）
        r1 = recent_results[t1][-WINDOW:]
        r2 = recent_results[t2][-WINDOW:]
        recent_wr1 = np.mean(r1) if r1 else 0.5
        recent_wr2 = np.mean(r2) if r2 else 0.5

        # 头对头胜率
        total_h2h = h2h_total.get(key, 0)
        if total_h2h > 0:
            h2h_rate = h2h_wins[key].get(t1, 0) / total_h2h
        else:
            h2h_rate = 0.5

        s1 = stats_map[t1]
        s2 = stats_map[t2]

        # 加入少量随机噪声模拟比赛不确定性
        noise = rng.normal(0, 0.015, size=6)

        rows.append({
            # ─ 基础排名特征 ─
            'rank_t1':            s1['rank'],
            'rank_t2':            s2['rank'],
            'rank_diff':          s2['rank'] - s1['rank'],         # 正值 = t1 更强
            'rank_ratio':         s1['rank'] / max(s2['rank'], 1),  # <1 = t1 更强

            # ─ 近期表现 ─
            'recent_wr_t1':       round(recent_wr1, 4),
            'recent_wr_t2':       round(recent_wr2, 4),
            'recent_wr_diff':     round(recent_wr1 - recent_wr2, 4),

            # ─ 头对头 ─
            'h2h_rate_t1':        round(h2h_rate, 4),

            # ─ 综合技术指标差 ─
            'rating_diff':        round(s1['rating']  - s2['rating']  + noise[0], 4),
            'kd_diff':            round(s1['kd_ratio'] - s2['kd_ratio'] + noise[1], 4),
            'kast_diff':          round(s1['kast']    - s2['kast']    + noise[2], 4),
            'adr_diff':           round(s1['adr']     - s2['adr']     + noise[3], 2),
            'opening_diff':       round(s1['opening_kill_rate'] - s2['opening_kill_rate'] + noise[4], 4),
            'clutch_diff':        round(s1['clutch_rate']       - s2['clutch_rate']       + noise[5], 4),
            'map_pool_diff':      s1['map_pool_depth'] - s2['map_pool_depth'],
            'flash_diff':         round(s1['flash_success'] - s2['flash_success'], 4),

            # ─ 绝对值特征 ─
            'rating_t1':          s1['rating'],
            'rating_t2':          s2['rating'],
            'adr_t1':             s1['adr'],
            'adr_t2':             s2['adr'],

            # ─ 标签 ─
            'team1':              t1,
            'team2':              t2,
            'result':             outcome,    # 1 = team1 胜
        })

        # 更新头对头和近期记录
        h2h_wins[key][t1] = h2h_wins[key].get(t1, 0) + outcome
        h2h_wins[key][t2] = h2h_wins[key].get(t2, 0) + (1 - outcome)
        h2h_total[key] += 1
        recent_results[t1].append(outcome)
        recent_results[t2].append(1 - outcome)

    return pd.DataFrame(rows)


df = generate_match_history(team_stats, n_matches=3000)
print(f'数据集形状：{df.shape}')
print(f'标签分布：\n{df["result"].value_counts()}')
df.head()

## 2. 探索性数据分析 (EDA)

In [ ]:
# ── 2.1 基本统计 ────────────────────────────────────────────────────────────
print('=== 缺失值检查 ===')
print(df.isnull().sum().sum(), '个缺失值')

print('\n=== 数值特征统计摘要 ===')
feature_cols = [c for c in df.columns if c not in ('team1', 'team2', 'result')]
df[feature_cols].describe().round(4)

In [ ]:
# ── 2.2 标签分布 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 标签饼图
counts = df['result'].value_counts()
axes[0].pie(counts, labels=['Team1 Win', 'Team2 Win'],
            autopct='%1.1f%%', startangle=90,
            colors=['#2196F3', '#FF5722'])
axes[0].set_title('比赛结果分布')

# 排名差与胜负
axes[1].hist([df[df['result']==1]['rank_diff'],
              df[df['result']==0]['rank_diff']],
             bins=30, alpha=0.7, label=['Team1 Win', 'Team2 Win'],
             color=['#2196F3', '#FF5722'])
axes[1].set_xlabel('排名差 (rank_t2 - rank_t1)')
axes[1].set_ylabel('场次')
axes[1].set_title('排名差 vs 比赛结果')
axes[1].legend()

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('图表已保存')

In [ ]:
# ── 2.3 特征相关性热力图 ────────────────────────────────────────────────────
key_features = ['rank_diff', 'rank_ratio', 'recent_wr_diff', 'h2h_rate_t1',
                'rating_diff', 'kd_diff', 'kast_diff', 'adr_diff',
                'opening_diff', 'clutch_diff', 'result']

plt.figure(figsize=(11, 9))
corr = df[key_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('特征相关性热力图', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.4 关键特征 Box-Plot ───────────────────────────────────────────────────
box_features = ['rating_diff', 'kd_diff', 'recent_wr_diff', 'adr_diff']
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
palette = {0: '#FF5722', 1: '#2196F3'}
labels = {0: 'Team2 Win', 1: 'Team1 Win'}

for ax, feat in zip(axes, box_features):
    for val in [0, 1]:
        data = df[df['result'] == val][feat]
        ax.boxplot(data, positions=[val], widths=0.4,
                   patch_artist=True,
                   boxprops=dict(facecolor=palette[val], alpha=0.7))
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['T2 Win', 'T1 Win'])
    ax.set_title(feat)

plt.suptitle('关键特征与比赛结果的关系', fontsize=13)
plt.tight_layout()
plt.savefig('boxplot_features.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. 特征工程与数据预处理

In [ ]:
# ── 3.1 构造额外交互特征 ────────────────────────────────────────────────────
df['composite_score_diff'] = (
    df['rating_diff'] * 2.0 +
    df['kd_diff']     * 1.5 +
    df['kast_diff']   * 1.0 +
    df['adr_diff']    / 20   # 归一化 adr
)

df['rank_rating_interaction'] = df['rank_diff'] * df['rating_diff']
df['form_rank_interaction']   = df['recent_wr_diff'] * (-df['rank_diff'])  # 近期状态 × 排名优势

FEATURE_COLS = [
    'rank_diff', 'rank_ratio',
    'recent_wr_t1', 'recent_wr_t2', 'recent_wr_diff',
    'h2h_rate_t1',
    'rating_diff', 'rating_t1', 'rating_t2',
    'kd_diff', 'kast_diff', 'adr_diff', 'adr_t1', 'adr_t2',
    'opening_diff', 'clutch_diff',
    'map_pool_diff', 'flash_diff',
    'composite_score_diff', 'rank_rating_interaction', 'form_rank_interaction',
]

X = df[FEATURE_COLS].copy()
y = df['result'].copy()

# 确保无 NaN（理论上模拟数据不会有，但防御性检查）
assert X.isnull().sum().sum() == 0, '存在缺失值，请检查数据生成逻辑'

print(f'特征数量：{X.shape[1]}')
print(f'样本数量：{X.shape[0]}')
print(f'标签正类比例：{y.mean():.4f}')

In [ ]:
# ── 3.2 训练/测试集划分 ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'训练集：{X_train.shape[0]} 样本  测试集：{X_test.shape[0]} 样本')

## 4. 基础模型训练与对比

In [ ]:
# ── 4.1 定义基础模型 ────────────────────────────────────────────────────────
base_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                        eval_metric='logloss', n_jobs=-1),
    'LightGBM':            LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                         verbose=-1, n_jobs=-1),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'KNN':                 KNeighborsClassifier(n_neighbors=9),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, model in base_models.items():
    # 线性模型使用标准化数据，树模型使用原始数据
    use_scaled = name in ('Logistic Regression', 'SVM', 'KNN')
    Xtr = X_train_sc if use_scaled else X_train.values
    Xte = X_test_sc  if use_scaled else X_test.values

    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv,
                                scoring='accuracy', n_jobs=-1)
    model.fit(Xtr, y_train)
    test_acc = accuracy_score(y_test, model.predict(Xte))
    test_auc = roc_auc_score(y_test, model.predict_proba(Xte)[:, 1])

    results[name] = {
        'model':    model,
        'scaled':   use_scaled,
        'cv_mean':  cv_scores.mean(),
        'cv_std':   cv_scores.std(),
        'test_acc': test_acc,
        'test_auc': test_auc,
    }
    print(f'{name:<22} CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  '
          f'TestAcc={test_acc:.4f}  AUC={test_auc:.4f}')

In [ ]:
# ── 4.2 模型对比可视化 ──────────────────────────────────────────────────────
model_names = list(results.keys())
cv_means    = [results[n]['cv_mean'] for n in model_names]
cv_stds     = [results[n]['cv_std']  for n in model_names]
test_accs   = [results[n]['test_acc'] for n in model_names]
test_aucs   = [results[n]['test_auc'] for n in model_names]

x = np.arange(len(model_names))
width = 0.28

fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x - width, cv_means,  width, label='CV Accuracy',   color='#1565C0', alpha=0.85)
b2 = ax.bar(x,          test_accs, width, label='Test Accuracy', color='#2E7D32', alpha=0.85)
b3 = ax.bar(x + width,  test_aucs, width, label='Test AUC-ROC', color='#B71C1C', alpha=0.85)

ax.errorbar(x - width, cv_means, yerr=cv_stds, fmt='none',
            ecolor='black', capsize=4, elinewidth=1.2)

ax.set_xlabel('模型')
ax.set_ylabel('分数')
ax.set_title('各基础模型性能对比', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha='right')
ax.set_ylim(0.5, 1.02)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 超参数调优（随机搜索 + 交叉验证）

In [ ]:
# ── 5.1 XGBoost 超参数搜索 ──────────────────────────────────────────────────
xgb_param_dist = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6, 7],
    'learning_rate':    [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample':        [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha':        [0, 0.01, 0.1, 0.5],
    'reg_lambda':       [0.5, 1.0, 1.5, 2.0],
    'min_child_weight': [1, 3, 5],
}

xgb_base = XGBClassifier(eval_metric='logloss',
                          random_state=RANDOM_STATE, n_jobs=-1)

xgb_search = RandomizedSearchCV(
    xgb_base, xgb_param_dist, n_iter=40, cv=cv,
    scoring='accuracy', random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
xgb_search.fit(X_train.values, y_train)

print('XGBoost 最优参数：')
print(xgb_search.best_params_)
print(f'最优 CV Accuracy：{xgb_search.best_score_:.4f}')

In [ ]:
# ── 5.2 LightGBM 超参数搜索 ─────────────────────────────────────────────────
lgb_param_dist = {
    'n_estimators':    [100, 200, 300, 500],
    'max_depth':       [3, 4, 5, 6, 8, -1],
    'learning_rate':   [0.01, 0.05, 0.1, 0.15],
    'num_leaves':      [15, 31, 63, 127],
    'min_child_samples': [5, 10, 20, 30],
    'subsample':       [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha':       [0, 0.01, 0.1],
    'reg_lambda':      [0, 0.1, 1.0],
}

lgb_base = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
lgb_search = RandomizedSearchCV(
    lgb_base, lgb_param_dist, n_iter=40, cv=cv,
    scoring='accuracy', random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
lgb_search.fit(X_train.values, y_train)

print('LightGBM 最优参数：')
print(lgb_search.best_params_)
print(f'最优 CV Accuracy：{lgb_search.best_score_:.4f}')

In [ ]:
# ── 5.3 Random Forest 超参数搜索 ────────────────────────────────────────────
rf_param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 5, 8, 12, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5, 0.8],
}

rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_base, rf_param_dist, n_iter=30, cv=cv,
    scoring='accuracy', random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
rf_search.fit(X_train.values, y_train)

print('Random Forest 最优参数：')
print(rf_search.best_params_)
print(f'最优 CV Accuracy：{rf_search.best_score_:.4f}')

## 6. Stacking 集成模型

In [ ]:
# ── 6.1 构建 Stacking 集成模型 ──────────────────────────────────────────────
best_xgb = xgb_search.best_estimator_
best_lgb = lgb_search.best_estimator_
best_rf  = rf_search.best_estimator_

# 第一层 (base learners)
estimators = [
    ('xgb', best_xgb),
    ('lgb', best_lgb),
    ('rf',  best_rf),
    ('gb',  GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                        max_depth=4, random_state=RANDOM_STATE)),
    ('lr',  Pipeline([('sc', StandardScaler()),
                      ('clf', LogisticRegression(max_iter=1000,
                                                 random_state=RANDOM_STATE))])),
]

# 第二层 (meta learner)：使用逻辑回归保证概率校准
meta_learner = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_learner,
    cv=5,
    passthrough=False,   # 不将原始特征传递给 meta learner
    n_jobs=-1,
)

stacking_model.fit(X_train.values, y_train)

stack_preds     = stacking_model.predict(X_test.values)
stack_proba     = stacking_model.predict_proba(X_test.values)[:, 1]
stack_acc       = accuracy_score(y_test, stack_preds)
stack_auc       = roc_auc_score(y_test, stack_proba)

print(f'Stacking 集成模型测试准确率：{stack_acc:.4f}')
print(f'Stacking 集成模型 AUC-ROC：  {stack_auc:.4f}')
print()
print('分类报告：')
print(classification_report(y_test, stack_preds,
                             target_names=['Team2 Win', 'Team1 Win']))

## 7. 综合评估

In [ ]:
# ── 7.1 混淆矩阵 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (preds, title) in zip(
        axes,
        [(best_xgb.predict(X_test.values), 'XGBoost (调优后)'),
         (stack_preds, 'Stacking 集成')]):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['T2 Win', 'T1 Win'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.suptitle('混淆矩阵对比', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.2 ROC 曲线对比 ─────────────────────────────────────────────────────────
models_to_plot = {
    'Logistic Regression': (results['Logistic Regression']['model'], True),
    'Random Forest (调优)': (best_rf, False),
    'XGBoost (调优)':       (best_xgb, False),
    'LightGBM (调优)':      (best_lgb, False),
    'Stacking 集成':        (stacking_model, False),
}

plt.figure(figsize=(8, 6))
colors = ['#607D8B', '#FF9800', '#2196F3', '#4CAF50', '#E91E63']

for (mname, (m, scaled)), color in zip(models_to_plot.items(), colors):
    Xte = X_test_sc if scaled else X_test.values
    proba = m.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{mname} (AUC={auc:.3f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='随机猜测')
plt.xlabel('假阳性率 (FPR)')
plt.ylabel('真阳性率 (TPR)')
plt.title('ROC 曲线对比', fontsize=13)
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.3 特征重要性（XGBoost） ───────────────────────────────────────────────
importances = best_xgb.feature_importances_
feat_imp_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True)

plt.figure(figsize=(9, 7))
plt.barh(feat_imp_df['feature'], feat_imp_df['importance'],
         color='#1565C0', alpha=0.85)
plt.xlabel('重要性分数')
plt.title('XGBoost 特征重要性', fontsize=13)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top-10 重要特征：')
print(feat_imp_df.sort_values('importance', ascending=False).head(10).to_string(index=False))

In [ ]:
# ── 7.4 综合性能汇总表 ──────────────────────────────────────────────────────
summary = []
for name, info in results.items():
    summary.append({
        '模型':     name,
        'CV Acc':   f"{info['cv_mean']:.4f} ± {info['cv_std']:.4f}",
        'Test Acc': f"{info['test_acc']:.4f}",
        'AUC-ROC':  f"{info['test_auc']:.4f}",
    })

# 加入调优后模型
for name, model, scaled in [
        ('XGBoost (调优)', best_xgb, False),
        ('LightGBM (调优)', best_lgb, False),
        ('Random Forest (调优)', best_rf, False),
        ('Stacking 集成', stacking_model, False),
]:
    Xte = X_test_sc if scaled else X_test.values
    cv_s = cross_val_score(model, X_train.values, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    summary.append({
        '模型':     name,
        'CV Acc':   f"{cv_s.mean():.4f} ± {cv_s.std():.4f}",
        'Test Acc': f"{accuracy_score(y_test, model.predict(Xte)):.4f}",
        'AUC-ROC':  f"{roc_auc_score(y_test, model.predict_proba(Xte)[:, 1]):.4f}",
    })

summary_df = pd.DataFrame(summary)
print('=== 模型性能总览 ===')
print(summary_df.to_string(index=False))

## 8. 比赛结果预测接口

In [ ]:
# ── 8.1 预测函数 ─────────────────────────────────────────────────────────────
def predict_match(team1_name: str, team2_name: str,
                  team_df: pd.DataFrame,
                  model, feature_cols: list,
                  recent_results_cache=None,  # type: Optional[dict]
                  h2h_cache=None,  # type: Optional[tuple]
                  verbose=True):
    """
    预测两队比赛结果。

    参数
    ----
    team1_name, team2_name : 队伍名称（必须在 team_df 中）
    team_df                : 队伍统计 DataFrame
    model                  : 已训练的分类器
    feature_cols           : 特征列列表
    recent_results_cache   : {team_name: [结果列表]} 近期表现缓存
    h2h_cache              : (h2h_wins_dict, h2h_total_dict) 头对头缓存
    verbose                : 是否打印预测详情

    返回
    ----
    dict 包含预测结果和概率
    """
    avail = set(team_df['team'])
    if team1_name not in avail:
        raise ValueError(f'"{team1_name}" 不在队伍列表中。\n可用：{sorted(avail)}')
    if team2_name not in avail:
        raise ValueError(f'"{team2_name}" 不在队伍列表中。\n可用：{sorted(avail)}')
    if team1_name == team2_name:
        raise ValueError('两队不能相同')

    stats_map = team_df.set_index('team').to_dict('index')
    s1 = stats_map[team1_name]
    s2 = stats_map[team2_name]

    # 近期胜率
    if recent_results_cache is not None:
        r1 = recent_results_cache.get(team1_name, [])[-15:]
        r2 = recent_results_cache.get(team2_name, [])[-15:]
    else:
        r1, r2 = [], []
    recent_wr1 = np.mean(r1) if r1 else 0.5
    recent_wr2 = np.mean(r2) if r2 else 0.5

    # 头对头
    if h2h_cache is not None:
        h2h_wins_d, h2h_total_d = h2h_cache
        key = tuple(sorted([team1_name, team2_name]))
        total_h2h = h2h_total_d.get(key, 0)
        h2h_rate = h2h_wins_d.get(key, {}).get(team1_name, 0) / total_h2h if total_h2h > 0 else 0.5
    else:
        h2h_rate = 0.5

    composite = (
        (s1['rating']  - s2['rating'])  * 2.0 +
        (s1['kd_ratio']- s2['kd_ratio'])* 1.5 +
        (s1['kast']    - s2['kast'])    * 1.0 +
        (s1['adr']     - s2['adr'])     / 20
    )

    feat_dict = {
        'rank_diff':              s2['rank'] - s1['rank'],
        'rank_ratio':             s1['rank'] / max(s2['rank'], 1),
        'recent_wr_t1':           recent_wr1,
        'recent_wr_t2':           recent_wr2,
        'recent_wr_diff':         recent_wr1 - recent_wr2,
        'h2h_rate_t1':            h2h_rate,
        'rating_diff':            s1['rating']  - s2['rating'],
        'rating_t1':              s1['rating'],
        'rating_t2':              s2['rating'],
        'kd_diff':                s1['kd_ratio'] - s2['kd_ratio'],
        'kast_diff':              s1['kast']    - s2['kast'],
        'adr_diff':               s1['adr']     - s2['adr'],
        'adr_t1':                 s1['adr'],
        'adr_t2':                 s2['adr'],
        'opening_diff':           s1['opening_kill_rate'] - s2['opening_kill_rate'],
        'clutch_diff':            s1['clutch_rate']       - s2['clutch_rate'],
        'map_pool_diff':          s1['map_pool_depth']    - s2['map_pool_depth'],
        'flash_diff':             s1['flash_success']     - s2['flash_success'],
        'composite_score_diff':   composite,
        'rank_rating_interaction': (s2['rank'] - s1['rank']) * (s1['rating'] - s2['rating']),
        'form_rank_interaction':  (recent_wr1 - recent_wr2) * (-(s2['rank'] - s1['rank'])),
    }

    X_pred = pd.DataFrame([feat_dict])[feature_cols]
    pred   = model.predict(X_pred.values)[0]
    proba  = model.predict_proba(X_pred.values)[0]

    winner     = team1_name if pred == 1 else team2_name
    win_prob   = proba[1] if pred == 1 else proba[0]
    t1_prob    = proba[1]
    t2_prob    = proba[0]

    if verbose:
        print('=' * 52)
        print(f'  {team1_name}  vs  {team2_name}')
        print('=' * 52)
        print(f'  排名：#{s1["rank"]}  vs  #{s2["rank"]}')
        print(f'  Rating：{s1["rating"]}  vs  {s2["rating"]}')
        print(f'  近期胜率：{recent_wr1:.1%}  vs  {recent_wr2:.1%}')
        print(f'  头对头胜率 ({team1_name})：{h2h_rate:.1%}')
        print('-' * 52)
        print(f'  预测胜者：🏆 {winner}')
        print(f'  {team1_name} 胜率：{t1_prob:.1%}')
        print(f'  {team2_name} 胜率：{t2_prob:.1%}')
        conf = abs(t1_prob - 0.5) / 0.5
        print(f'  置信度：{conf:.1%}')
        print('=' * 52)

    return {
        'team1': team1_name, 'team2': team2_name,
        'predicted_winner': winner,
        'team1_win_prob': round(t1_prob, 4),
        'team2_win_prob': round(t2_prob, 4),
        'confidence': round(abs(t1_prob - 0.5) / 0.5, 4),
    }


# ── 8.2 测试预测接口 ─────────────────────────────────────────────────────────
print('【测试预测 1】')
r1 = predict_match('Natus Vincere', 'Astralis',
                   team_stats, stacking_model, FEATURE_COLS)

print('\n【测试预测 2】')
r2 = predict_match('FaZe Clan', 'FURIA',
                   team_stats, stacking_model, FEATURE_COLS)

print('\n【测试预测 3 – 势均力敌】')
r3 = predict_match('G2 Esports', 'FaZe Clan',
                   team_stats, stacking_model, FEATURE_COLS)

In [ ]:
# ── 8.3 概率可视化 ───────────────────────────────────────────────────────────
matchups = [
    ('Natus Vincere', 'Astralis'),
    ('G2 Esports', 'FaZe Clan'),
    ('Heroic', 'ENCE'),
    ('Liquid', 'Cloud9'),
    ('MOUZ', 'BIG'),
    ('Vitality', 'NIP'),
]

matchup_results = []
for t1, t2 in matchups:
    res = predict_match(t1, t2, team_stats, stacking_model, FEATURE_COLS, verbose=False)
    matchup_results.append(res)

labels   = [f"{r['team1']}\nvs\n{r['team2']}" for r in matchup_results]
t1_probs = [r['team1_win_prob'] for r in matchup_results]
t2_probs = [r['team2_win_prob'] for r in matchup_results]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x, t1_probs, label='Team1 胜率', color='#1565C0', alpha=0.85)
ax.bar(x, t2_probs, bottom=t1_probs, label='Team2 胜率',
       color='#B71C1C', alpha=0.85)
ax.axhline(0.5, color='white', linestyle='--', linewidth=1.5, label='50% 基准线')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('胜率')
ax.set_title('多场对决胜率预测', fontsize=13)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('win_probability.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. 结论与改进点总结

| 方面 | 本项目实现 |
|------|------------|
| **数据特征** | 排名、近期胜率、头对头、技术指标（Rating/KD/KAST/ADR/Opening/Clutch）+ 交互特征 |
| **模型** | LR / RF / XGBoost / LightGBM / SVM / KNN + Stacking 集成 |
| **调优** | 随机搜索 + 5-折交叉验证 |
| **评估** | Accuracy、AUC-ROC、混淆矩阵、分类报告 |
| **预测接口** | `predict_match()` 函数，输出胜者、概率、置信度 |

### 进一步提升准确率的方向
1. **真实数据接入** – 使用 HLTV API 或 Kaggle 数据集替换模拟数据
2. **地图维度** – 针对不同地图分别建模（各地图胜率差异显著）
3. **时序模型** – 使用 LSTM/Transformer 捕捉队伍形态的时序演变
4. **选手级别特征** – 将五名选手的个人 Rating 聚合为队伍特征
5. **赛事权重** – Major > ESL Pro League > 普通在线赛，加权样本
6. **贝叶斯优化** – 用 Optuna 替代随机搜索进行超参数调优